Before running this notbook, please run codonaligner.ipynb and filting_introns.ipynb

In [5]:
import cogent3
from cogent3 import get_app
from cogent3.maths.matrix_exponential_integration import expected_number_subs
import matplotlib.pyplot as plt
import paths
import libs
import pandas as pd
import numpy as np
import pickle
from tqdm.notebook import tqdm
import os

import trinuc_models as trinucs # this module must be in the same directory as this notebook

# Non cds

In [6]:
#Setting up the rules for model fitting of noncds regions
sm_noncds=trinucs.GT_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_noncds = [{"par_name": n, "is_independent": True} for n in paramnames if n!="omega"] + [{"par_name": "omega", "value": 1.0, "is_constant": True}]
GT_subsmodel = get_app("model", "GT_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      param_rules=rules_noncds)

#Setting up the app for reading noncds sequences
#rename_noncds eliminates files that do not contain Orangutan, Human and Chimp sequences or that has duplicates.
loader = get_app("load_aligned", moltype="dna")
rename_noncds = libs.renamer_noncds_aligned()
#Need to omit gapped positions. My alignmetns is a subset of 10 primate alignments.
#If a indel spread across all 10 species, then the position is not informative and can be omitted.
omit_gap_pos_app = get_app("omit_gap_pos", moltype="dna")
#Because I am using a trinucleotide I need to omit degenerates using motif 3
omit_degs_noncds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

noncds_app = loader + rename_noncds + omit_gap_pos_app + omit_degs_noncds

## Intergenic Ancestral Repeats (IGAR)

In [8]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'intergenicAR/chrm10/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGAR10 = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
alns_IGAR10 = concat(nonconcat_IGAR10)

folder_in = paths.DATA_HUMCHIMPORANG115 + 'intergenicAR/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGAR22 = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
alns_IGAR22 = concat(nonconcat_IGAR22)

alns_IGAR = concat([alns_IGAR10, alns_IGAR22])
print("alignment length:", len(alns_IGAR))
alns_IGAR.source = "IGAR_alignments"


   0%|          |00:00<?

   0%|          |00:00<?

alignment length: 88026


In [10]:

folder_out = paths.DATA_HUMCHIMPORANG115 + 'intergenicAR/trinucleotide_filtered_chrm22_10/'
out_dstore = cogent3.open_data_store(folder_out, suffix='fa', mode='r')

# 3. Create the writer app
write_seqs_app = get_app("write_seqs", data_store=out_dstore, format_name="fasta")

# 4. Write the alignment
_ = write_seqs_app(alns_IGAR)

OSError: '/xdisk/masel/uliseshmc/EstimatingUd/HumChimOran_10_115/intergenicAR/trinfiltered_chrm22_10' does not exist

In [9]:
result_IGAR = GT_subsmodel(alns_IGAR)

result_IGAR.lf

/home/u12/uliseshmc/.conda/envs/UdChimpHumOran/lib/python3.13/site-packages/cogent3/recalculation/definition.py:690: UserWarning: using slow exponentiator because 'eigen failed precision test'
  return func(*args)


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -340296.5845
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Chimpanzee    root        2.18              50.00    0.07    0.08    0.00
Orangutan     root        3.36              21.46    0.49    0.50    0.57
Human         root        2.12              50.00    0.00    0.02    0.00
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.06    0.06    0.14    0.07    0.12    0.00    1.19    1.08
0.41    0.50    0.56    0.39    0.55    0.51    1.30    1.06
0.02    0.00    0.14    0.07    0.02    0.00    1.12    1.10
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.01    0.00    0.01    0.01    0.01    0.00    0.00    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.01    0.02    0.00    0.00    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.00    0.00    0.01    0.00    0.00    0.00    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.02    0.02    0.01    0.00    0.01    0.01    0.01    0.00    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.00    0.00    0.01    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.01    0.01    0.01    0.03    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.01    0.01    0.52
----------------------------

## Introns

In [ ]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'introns/chrm10/noUTRs/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_introns = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_introns = concat(nonconcat_introns)
print("alignment length:", len(alns_introns))
alns_introns.source = "introns_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_introns = GT_subsmodel(alns_introns)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_introns.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_introns))

result_introns.lf

   0%|          |00:00<?

alignment length: 83646


/home/u12/uliseshmc/.conda/envs/UdChimpHumOran/lib/python3.13/site-packages/cogent3/recalculation/definition.py:690: UserWarning: using slow exponentiator because 'eigen failed precision test'
  return func(*args)


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -314560.8225
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        2.05             100.00    0.00    0.05    0.00
Orangutan     root        3.14              17.89    0.36    0.44    0.46
Chimpanzee    root        2.06             100.00    0.00    0.05    0.00
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.02    0.09    0.00    0.02    0.00    1.38    1.08
0.35    0.35    0.67    0.19    0.50    0.50    1.51    1.09
0.00    0.03    0.10    0.00    0.00    0.00    1.43    1.14
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.01    0.00    0.01    0.01    0.01    0.00    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.00    0.02    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.00    0.03    0.01    0.00    0.01    0.00    0.00    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.00    0.00    0.00    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.01    0.01    0.01    0.03    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.02    0.01    0.00    0.51
----------------------------

## Introns Ancestral Repeats (IntronAR)

In [ ]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'intronsAR/chrm10/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_intronsAR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_intronsAR = concat(nonconcat_intronsAR)
print("alignment length:", len(alns_intronsAR))
alns_intronsAR.source = "intronsAR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_intronsAR = GT_subsmodel(alns_intronsAR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_intronsAR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_intronsAR))

result_intronsAR.lf

   0%|          |00:00<?

alignment length: 73164


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -271342.1809
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        2.03             100.00    0.00    0.03    0.00
Orangutan     root        2.81              17.49    0.31    0.33    0.29
Chimpanzee    root        2.01             100.00    0.01    0.05    0.00
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.00    0.07    0.00    0.02    0.00    1.11    1.00
0.28    0.29    0.48    0.13    0.38    0.35    1.25    1.02
0.00    0.03    0.08    0.00    0.02    0.00    1.24    1.04
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.01    0.00    0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.00    0.02    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.02    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.00    0.02    0.01    0.00    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.00    0.00    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.01    0.01    0.02    0.03    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.02    0.01    0.00    0.48
----------------------------